# RFC Simulator and Paper-Reproduction Notebook

This notebook is a minimal Binder-safe reader and runner for the RFC notebook package.

It intentionally uses only standard Python plus the notebook display system. It does not require any scientific Python packages.

Expected files, either in the repository root or in `notebooks/`:

- `SimulationConfigs.json`
- `Module_G_R_N_S_T_FrozenPacket.json`
- `ValidationScreens_U_V_W_X_Y2_Z_QG.json`

Run order:

`G -> R -> N -> DownstreamPhysicalProjection -> DimensionlessValidation -> S -> T -> U -> V -> W -> X -> Y2 -> Z -> QG`

Legacy/development modules A-Q plus `G_legacy`, `N_legacy`, and `R_legacy` are supported when present in `SimulationConfigs.json`.


In [ ]:
import json
import math
from pathlib import Path

try:
    from IPython.display import display, Markdown, HTML
except Exception:
    def display(x):
        print(x)
    def Markdown(x):
        return x
    def HTML(x):
        return x

BASE_DIR = Path.cwd()

EXPECTED_FILES = {
    "simulation_configs": "SimulationConfigs.json",
    "frozen_packet": "Module_G_R_N_S_T_FrozenPacket.json",
    "validation_screens": "ValidationScreens_U_V_W_X_Y2_Z_QG.json"
}

PAPER_REPRODUCTION_ORDER = [
    "G", "R", "N", "DownstreamPhysicalProjection", "DimensionlessValidation",
    "S", "T", "U", "V", "W", "X", "Y2", "Z", "QG"
]

LEGACY_DEVELOPMENT_ORDER = [
    "A", "B", "C", "D", "E", "F", "H", "I", "J", "K", "L", "M", "O", "P", "Q",
    "G_legacy", "N_legacy", "R_legacy"
]

LEGACY_MODULES = {"G_legacy", "N_legacy", "R_legacy"}

print("Working directory:", BASE_DIR)
print("RFC simulator mode: standard-library reader; no external scientific packages required.")


In [ ]:
def escape_html(value):
    text = str(value)
    return (
        text.replace("&", "&amp;")
            .replace("<", "&lt;")
            .replace(">", "&gt;")
            .replace('"', "&quot;")
    )


def compact_value(value, max_len=1200):
    if isinstance(value, (dict, list)):
        text = json.dumps(value, ensure_ascii=False, indent=2)
    else:
        text = str(value)
    if len(text) > max_len:
        return text[:max_len] + " ..."
    return text


def display_title(title):
    display(Markdown(f"## {title}"))


def display_note(text):
    display(Markdown(text))


def display_table(rows, max_rows=200):
    if rows is None:
        display_note("`None`")
        return
    if isinstance(rows, dict):
        rows = [{"field": k, "value": v} for k, v in rows.items()]
    if not isinstance(rows, list):
        display_note("`" + escape_html(compact_value(rows)) + "`")
        return
    if len(rows) == 0:
        display_note("_No rows to display._")
        return

    normalized = []
    for item in rows[:max_rows]:
        if isinstance(item, dict):
            normalized.append(item)
        else:
            normalized.append({"value": item})

    columns = []
    for row in normalized:
        for key in row.keys():
            if key not in columns:
                columns.append(key)

    html = []
    html.append("<div style='overflow-x:auto; max-width:100%;'>")
    html.append("<table style='border-collapse:collapse; width:100%; font-size:14px;'>")
    html.append("<thead><tr>")
    for col in columns:
        html.append("<th style='border:1px solid #bbb; padding:6px; background:#f2f2f2; text-align:left;'>" + escape_html(col) + "</th>")
    html.append("</tr></thead><tbody>")
    for row in normalized:
        html.append("<tr>")
        for col in columns:
            value = compact_value(row.get(col, ""))
            html.append("<td style='border:1px solid #bbb; padding:6px; vertical-align:top; white-space:pre-wrap;'>" + escape_html(value) + "</td>")
        html.append("</tr>")
    html.append("</tbody></table></div>")
    if len(rows) > max_rows:
        html.append(f"<p>Showing first {max_rows} rows of {len(rows)} total rows.</p>")
    display(HTML("".join(html)))


In [ ]:
def candidate_paths(filename):
    cwd = Path.cwd()
    roots = [cwd, cwd / "notebooks", cwd.parent, cwd.parent / "notebooks"]
    paths = [root / filename for root in roots]
    try:
        paths.extend(cwd.rglob(filename))
    except Exception:
        pass
    unique = []
    seen = set()
    for path in paths:
        key = str(path)
        if key not in seen:
            seen.add(key)
            unique.append(path)
    return unique


def locate_file(filename):
    for path in candidate_paths(filename):
        if path.exists() and path.is_file():
            return path
    return None


def load_json_file(filename):
    path = locate_file(filename)
    if path is None:
        return None, {"file": filename, "path": "not found", "exists": False, "loaded": False, "error": "not found"}
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data, {"file": filename, "path": str(path), "exists": True, "loaded": True, "error": ""}
    except Exception as exc:
        return None, {"file": filename, "path": str(path), "exists": True, "loaded": False, "error": str(exc)}


simulation_configs, simulation_status = load_json_file(EXPECTED_FILES["simulation_configs"])
frozen_packet, frozen_status = load_json_file(EXPECTED_FILES["frozen_packet"])
validation_file, validation_status = load_json_file(EXPECTED_FILES["validation_screens"])

display_title("1. File load status")
display_table([simulation_status, frozen_status, validation_status])

if frozen_packet is None and simulation_configs is None:
    display_note("**ERROR:** No usable RFC JSON files were found. Put the JSON files in the repository root or in `notebooks/`.")
else:
    display_note("Loaded available RFC files. Continuing with schema normalization.")


In [ ]:
def as_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]


def normalize_name(name):
    return str(name).lower().replace("_", "").replace("-", "").replace(" ", "")


def unwrap_modules(config):
    if config is None:
        return []
    if isinstance(config, list):
        return config
    if isinstance(config, dict):
        for key in ["modules", "moduleConfigs", "simulationModules", "configs"]:
            if isinstance(config.get(key), list):
                return config[key]
    return []


simulation_modules = unwrap_modules(simulation_configs)


def module_name(module_obj):
    if not isinstance(module_obj, dict):
        return None
    for key in ["module", "id", "name", "moduleId", "moduleID"]:
        value = module_obj.get(key)
        if isinstance(value, str):
            return value
    return None


def available_config_module_names():
    names = []
    for obj in simulation_modules:
        name = module_name(obj)
        if name is not None:
            names.append(name)
    return names


def find_module_in_configs(name):
    target = normalize_name(name)
    for obj in simulation_modules:
        obj_name = module_name(obj)
        if obj_name and normalize_name(obj_name) == target:
            return obj
    return None


def find_key_contains(obj, tokens):
    if not isinstance(obj, dict):
        return None
    tokens_low = [str(t).lower() for t in tokens]
    for key, value in obj.items():
        key_low = str(key).lower()
        if all(t in key_low for t in tokens_low):
            return value
    return None


def get_packet_module(name):
    if not isinstance(frozen_packet, dict):
        return None
    direct = {
        "G": ["moduleG", "G"],
        "R": ["moduleR", "R"],
        "N": ["moduleN_V2", "moduleN", "N"],
        "S": ["moduleS", "S"],
        "T": ["moduleT", "T"],
        "DownstreamPhysicalProjection": ["downstreamPhysicalProjectionScreen", "downstreamPhysicalProjection", "DownstreamPhysicalProjection"],
        "DimensionlessValidation": ["dimensionlessValidationLayer", "dimensionlessValidation", "DimensionlessValidation"]
    }
    for key in direct.get(name, [name]):
        if key in frozen_packet:
            return frozen_packet[key]

    target = normalize_name(name)
    for key, value in frozen_packet.items():
        key_clean = normalize_name(key)
        if key_clean == target or key_clean == normalize_name("module" + str(name)):
            return value
    return None


def raw_validation_block():
    if isinstance(validation_file, dict) and isinstance(validation_file.get("validationScreens"), dict):
        return validation_file["validationScreens"]
    if isinstance(validation_file, dict):
        return validation_file
    if isinstance(frozen_packet, dict) and isinstance(frozen_packet.get("validationScreens"), dict):
        return frozen_packet["validationScreens"]
    return {}


validation_screens = raw_validation_block()


def get_validation_screen(name):
    if isinstance(validation_screens, dict):
        if name in validation_screens:
            return validation_screens[name]
        target = normalize_name(name)
        for key, value in validation_screens.items():
            if normalize_name(key) == target:
                return value
    return find_module_in_configs(name)


def get_module_data(name):
    if name in ["U", "V", "W", "X", "Y2", "Z", "QG"]:
        return get_validation_screen(name)
    if name in ["G", "R", "N", "S", "T", "DownstreamPhysicalProjection", "DimensionlessValidation"]:
        return get_packet_module(name) or find_module_in_configs(name)
    return find_module_in_configs(name) or get_packet_module(name)


def build_all_runnable_modules():
    ordered = []
    for name in PAPER_REPRODUCTION_ORDER + LEGACY_DEVELOPMENT_ORDER:
        if name not in ordered:
            ordered.append(name)
    for name in available_config_module_names():
        if name not in ordered:
            ordered.append(name)
    return ordered


ALL_RUNNABLE_MODULES = build_all_runnable_modules()


In [ ]:
def flatten_dict(obj, prefix=""):
    rows = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            field = f"{prefix}.{key}" if prefix else str(key)
            if isinstance(value, dict):
                rows.extend(flatten_dict(value, field))
            elif isinstance(value, list):
                if all(isinstance(item, dict) for item in value):
                    rows.append({"field": field, "value": f"list[{len(value)}] of records"})
                else:
                    rows.append({"field": field, "value": value})
            else:
                rows.append({"field": field, "value": value})
    else:
        rows.append({"field": prefix or "value", "value": obj})
    return rows


def first_record_list(obj):
    if isinstance(obj, list) and all(isinstance(item, dict) for item in obj):
        return obj
    if isinstance(obj, dict):
        for key in ["results", "screenResults", "selectedResults", "scoredConstants", "constants", "rows", "quarks", "mixingAngles", "observerBranchingResults", "neuralEEGTargets"]:
            if key in obj:
                found = first_record_list(obj[key])
                if found is not None:
                    return found
        for value in obj.values():
            found = first_record_list(value)
            if found is not None:
                return found
    return None


def display_object(obj, title="Object"):
    display_title(title)
    if obj is None:
        display_note("**Not found.** Check file names and file locations.")
        return

    if isinstance(obj, dict):
        for key in ["description", "currentStatus", "status", "runMode", "claimBoundary", "boundary", "interpretation"]:
            if key in obj:
                display_note(f"**{key}:** {compact_value(obj[key], 2000)}")

    records = first_record_list(obj)
    if records:
        display_table(records)

    if isinstance(obj, dict):
        display_table(flatten_dict(obj))
    elif isinstance(obj, list):
        display_table(obj)
    else:
        display_note("`" + escape_html(compact_value(obj)) + "`")


def numeric_or_none(value):
    try:
        if isinstance(value, bool):
            return None
        return float(value)
    except Exception:
        return None


def deep_get_by_key(obj, keys):
    targets = {str(k).lower() for k in as_list(keys)}
    if isinstance(obj, dict):
        for key, value in obj.items():
            if str(key).lower() in targets:
                return value
        for value in obj.values():
            found = deep_get_by_key(value, keys)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for value in obj:
            found = deep_get_by_key(value, keys)
            if found is not None:
                return found
    return None


In [ ]:
def module_g_check():
    data = get_module_data("G")
    display_title("Module G: Deterministic Triadic Closure")
    if data is None:
        display_note("**Module G not found.**")
        return

    delta = numeric_or_none(deep_get_by_key(data, ["delta"]))
    cycle_length = numeric_or_none(deep_get_by_key(data, ["cycleLength", "cycle_length"]))
    phase_depth_k = numeric_or_none(deep_get_by_key(data, ["phaseDepthK", "phase_depth_k"]))
    alpha_packet = numeric_or_none(deep_get_by_key(data, ["alpha"]))
    nu_packet = numeric_or_none(deep_get_by_key(data, ["nu"]))
    epsilon_packet = numeric_or_none(deep_get_by_key(data, ["epsilon"]))
    empirical_targets = deep_get_by_key(data, ["empiricalTargetsUsed", "empirical_targets_used"])

    rows = []
    if delta is not None and cycle_length is not None and alpha_packet is not None:
        expected = math.log(delta) / cycle_length
        rows.append({"check": "alpha = log(delta) / cycleLength", "expected": expected, "packet": alpha_packet, "absError": abs(expected - alpha_packet)})
    if delta is not None and phase_depth_k is not None and nu_packet is not None:
        expected = phase_depth_k * delta ** (-4)
        rows.append({"check": "nu = phaseDepthK * delta^(-4)", "expected": expected, "packet": nu_packet, "absError": abs(expected - nu_packet)})
    if alpha_packet is not None and nu_packet is not None and epsilon_packet is not None:
        expected = alpha_packet * nu_packet
        rows.append({"check": "epsilon = alpha * nu", "expected": expected, "packet": epsilon_packet, "absError": abs(expected - epsilon_packet)})
    rows.append({"check": "empiricalTargetsUsed", "expected": False, "packet": empirical_targets, "status": "PASS" if empirical_targets is False else "CHECK"})

    display_table(rows)
    display_object(data, "Module G stored data")


def module_r_check():
    data = get_module_data("R")
    display_title("Module R: Triad-Grouped Global Closure Audit")
    if data is None:
        display_note("**Module R not found.**")
        return

    keys = [
        "rawRFLResidualScore", "sourceCoupledRFLResidualScore", "residualImprovement",
        "rawStandardizedResidualScore", "sourceCoupledRFLResidualScoreV2", "residualImprovementV2",
        "moduleRScoreV2", "cpResidual", "tailN18", "tailN40", "bestLagCorrelation"
    ]
    rows = []
    for key in keys:
        value = deep_get_by_key(data, [key])
        if value is not None:
            rows.append({"field": key, "value": value})
    display_table(rows)

    placeholder_flags = []
    for key, bad in [("rawRFLResidualScore", 0.5), ("sourceCoupledRFLResidualScore", 0.5), ("residualImprovement", 0.0), ("tailN18", 0.0), ("tailN40", 0.0)]:
        value = numeric_or_none(deep_get_by_key(data, [key]))
        if value is not None and abs(value - bad) < 1e-15:
            placeholder_flags.append(key)
    if placeholder_flags:
        display_note("**WARNING:** possible placeholder Module R values detected: " + ", ".join(placeholder_flags))
    else:
        display_note("No obvious placeholder Module R values detected.")

    display_object(data, "Module R stored data")


def run_component(name):
    if name == "G":
        module_g_check()
    elif name == "R":
        module_r_check()
    elif name == "N":
        display_object(get_module_data("N"), "Module N V2: Dimensional Projection Bridge")
    elif name == "S":
        display_object(get_module_data("S"), "Module S: One-Anchor SI Bridge")
    elif name == "T":
        display_object(get_module_data("T"), "Module T: Dimensionless Coupling Map")
    elif name == "DownstreamPhysicalProjection":
        display_object(get_module_data(name), "Downstream Physical-Projection Screen")
    elif name == "DimensionlessValidation":
        display_object(get_module_data(name), "Dimensionless Observable Validation Layer")
    elif name in ["U", "V", "W", "X", "Y2", "Z", "QG"]:
        titles = {
            "U": "Module U: One-Anchor Constant Table Screen",
            "V": "Module V: Precision Cosmology Compressed-Parameter Screen",
            "W": "Module W: BBN Light-Abundance Proxy Screen",
            "X": "Module X: CP/EDM Bound Screen",
            "Y2": "Module Y2: Particle-Sector Refinement Screen",
            "Z": "Module Z: Observer, Branching, Neural, and EEG Harness",
            "QG": "Module QG: Finite Spin-Foam Transition-Amplitude Audit"
        }
        display_object(get_module_data(name), titles.get(name, f"Module {name}"))
        if name == "Y2":
            display_note("**Boundary:** Y2 is exploratory candidate discovery. It is not independent validation until frozen and retested as Y3.")
    else:
        display_object(get_module_data(name), f"Module {name}")
        if name in LEGACY_MODULES or name in LEGACY_DEVELOPMENT_ORDER:
            display_note("**Legacy/development note:** retained for continuity; not part of the active paper-reproduction spine unless separately stated.")


def run_all_paper():
    display(Markdown("# RFC Full Paper-Reproduction Pass"))
    for name in PAPER_REPRODUCTION_ORDER:
        run_component(name)
        display(HTML("<hr>"))


def run_legacy_development_modules():
    display(Markdown("# RFC Legacy / Development Simulator Pass"))
    found_any = False
    for name in LEGACY_DEVELOPMENT_ORDER:
        if get_module_data(name) is not None:
            found_any = True
            run_component(name)
            display(HTML("<hr>"))
    if not found_any:
        display_note("No legacy/development modules were found in SimulationConfigs.json.")


def run_all():
    run_all_paper()


In [ ]:
def package_audit():
    display_title("2. Repository Package Audit")
    rows = []
    rows.append({"check": "SimulationConfigs.json loaded", "status": simulation_configs is not None})
    rows.append({"check": "Frozen packet JSON loaded", "status": frozen_packet is not None})
    rows.append({"check": "Standalone validation screens JSON loaded", "status": validation_file is not None})

    for name in PAPER_REPRODUCTION_ORDER:
        rows.append({"path": "paper", "component": name, "found": get_module_data(name) is not None})
    for name in LEGACY_DEVELOPMENT_ORDER:
        rows.append({"path": "legacy", "component": name, "found": get_module_data(name) is not None})

    display_table(rows)

    missing_paper = [row["component"] for row in rows if row.get("path") == "paper" and row.get("found") is False]
    if missing_paper:
        display_note("**Missing paper-reproduction components:** " + ", ".join(missing_paper))
    else:
        display_note("**PASS:** all paper-reproduction components resolved.")

    available_legacy = [row["component"] for row in rows if row.get("path") == "legacy" and row.get("found") is True]
    if available_legacy:
        display_note("Legacy/development modules available: " + ", ".join(available_legacy))


package_audit()


## Manual commands

Use these commands after running the cells above:

```python
package_audit()
run_all_paper()
run_component("G")
run_component("R")
run_component("N")
run_component("DownstreamPhysicalProjection")
run_component("DimensionlessValidation")
run_component("S")
run_component("T")
run_component("U")
run_component("V")
run_component("W")
run_component("X")
run_component("Y2")
run_component("Z")
run_component("QG")
run_legacy_development_modules()
```

Interpretation boundaries:

- Module G is the frozen packet source.
- R/N/S/T and downstream screens consume the frozen packet.
- Validation screens are audit/comparison screens, not retuning mechanisms.
- Y2 is exploratory candidate discovery and must be frozen/retested as Y3 before independent validation claims.
- W is a BBN light-abundance proxy screen, not a full reaction network.
- QG is a finite spin-foam audit, not a complete proof of quantum gravity.
